# Q10 v1 — Validacao rigorosa K=11 (10 baseline + mode_bin)

**Metodologia:** split Train(70%) / Val(15%) / Test(15%) ANTES de qualquer fit. Refit K=10 e K=11 em Train. Decisao em Val. Test so no final.

**Por que rigoroso?** Para que a decisao entre K=10 e K=11 nao seja contaminada por informacao de Test. Treinamos em Train, comparamos em Val, e Test serve apenas para a avaliacao final do modelo escolhido.

**Tempo estimado no Colab T4:** ~55 min (2 fits ADVI + LOO + Val + Test).

**Outputs salvos em** `relatorio/analises/resultados/q10_*.{nc,csv,txt}`


In [ ]:
# C2 — Install
!pip install -q pymc==6.3.1 arviz==1.3.0 numpyro jax[cuda12] h5py h5netcdf pandas pyarrow scipy scikit-learn


In [ ]:
# C3 — Verify T4 GPU
!nvidia-smi


In [ ]:
# C4 — Mount Drive + paths
import os
from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = '/content/drive/MyDrive/spotify_challenge/insights-spotfy-grupo-4'
DATA_PARQUET = f'{PROJECT_ROOT}/data/processed/spotify_tracks_limpo.parquet'
RESULTS_DIR = f'{PROJECT_ROOT}/relatorio/analises/resultados'
os.makedirs(RESULTS_DIR, exist_ok=True)
print('RESULTS_DIR:', RESULTS_DIR)

import numpy as np
import pandas as pd
import pymc as pm
import arviz as az
import time
import scipy.sparse
from sklearn.metrics import roc_auc_score, brier_score_loss, log_loss

SEED = 42
np.random.seed(SEED)
print('SEED:', SEED)


In [ ]:
# C5 — Load parquet + clean non-musical genres
df = pd.read_parquet(DATA_PARQUET)
print(f'n faixas carregadas: {len(df):,}')

NON_MUSICAL = ['sleep', 'study', 'comedy', 'kids', 'children', 'new-age']
def is_non_music(s):
    if pd.isna(s): return False
    return any(g in NON_MUSICAL for g in str(s).lower().split())

mask_non = df['generos'].apply(is_non_music)
df = df[~mask_non].copy().reset_index(drop=True)
print(f'apos cleaning: {len(df):,}')

BASELINE_FEATS = ['danceability', 'energy', 'loudness', 'speechiness',
                  'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'explicit']
K11_FEATS = BASELINE_FEATS + ['mode_bin']
print(f'BASELINE_FEATS: {len(BASELINE_FEATS)} features')
print(f'K11_FEATS: {len(K11_FEATS)} features (10 baseline + mode_bin)')


In [ ]:
# C6 — Engineering: mode_bin + explicit + z-score
df['mode_bin'] = df['mode'].astype(int)
df['explicit'] = df['explicit'].astype(int)

cont_feats = [f for f in K11_FEATS if f not in ('explicit', 'mode_bin')]
means = df[cont_feats].mean()
stds = df[cont_feats].std()
df[cont_feats] = (df[cont_feats] - means) / stds

df = df.dropna(subset=K11_FEATS + ['popularity', 'genero_principal']).reset_index(drop=True)
print(f'apos dropna: {len(df):,}')

FULL_X = df[K11_FEATS].values.astype('float32')
print(f'FULL_X shape: {FULL_X.shape}')

y_pop = df['popularity'].values.astype('float32')
thresh = np.quantile(y_pop[y_pop > 0], 0.75)
FULL_y_top25 = (y_pop >= thresh).astype('int32')
print(f'thresh top25: {thresh:.1f} | pos rate: {FULL_y_top25.mean():.3f}')

genero_cats = sorted(df['genero_principal'].unique())
genero_to_idx = {g: i for i, g in enumerate(genero_cats)}
FULL_g_idx = df['genero_principal'].map(genero_to_idx).astype('int32').values
n_generos = len(genero_cats)
print(f'n_generos: {n_generos}')


In [ ]:
# C7 — *** THE CRITICAL SPLIT *** Train/Val/Test com SEED fixo
rng = np.random.default_rng(SEED)
N = len(df)
idx = np.arange(N)
rng.shuffle(idx)

n_train = int(0.70 * N)
n_val = int(0.15 * N)
# n_test = N - n_train - n_val

train_idx = idx[:n_train]
val_idx = idx[n_train:n_train + n_val]
test_idx = idx[n_train + n_val:]

print('=== SPLIT ===')
print(f'Train: {len(train_idx):,} ({len(train_idx)/N:.1%})')
print(f'Val:   {len(val_idx):,} ({len(val_idx)/N:.1%})')
print(f'Test:  {len(test_idx):,} ({len(test_idx)/N:.1%})')
print(f'Soma:  {len(train_idx) + len(val_idx) + len(test_idx):,} (de {N:,})')

# Confirmar sem sobreposicao
assert len(set(train_idx) & set(val_idx)) == 0, 'train/val overlap!'
assert len(set(train_idx) & set(test_idx)) == 0, 'train/test overlap!'
assert len(set(val_idx) & set(test_idx)) == 0, 'val/test overlap!'
print('Sem sobreposicao entre Train/Val/Test. OK.')

# Construir matrizes de cada split
X_train = FULL_X[train_idx]
X_val   = FULL_X[val_idx]
X_test  = FULL_X[test_idx]
y_train = FULL_y_top25[train_idx]
y_val   = FULL_y_top25[val_idx]
y_test  = FULL_y_top25[test_idx]
g_train = FULL_g_idx[train_idx]
g_val   = FULL_g_idx[val_idx]
g_test  = FULL_g_idx[test_idx]

# Sanity: prevalencia similar entre splits
print(f'\nPrevalencia top-25: train={y_train.mean():.3f}, val={y_val.mean():.3f}, test={y_test.mean():.3f}')


## 8. Sanity check do split

Verificacoes:
- Sem sobreposicao entre Train, Val e Test (assert em C7)
- Prevalencia de top-25 similar nos 3 splits (~25% cada, como esperado)
- Split por shuffle aleatorio com SEED=42 (reproduzivel)

**REGRA DE OURO daqui pra frente:**
- `X_train`, `y_train`, `g_train` → somente para FITAR
- `X_val`, `y_val`, `g_val` → somente para DECIDIR entre K=10 e K=11
- `X_test`, `y_test`, `g_test` → TOCA UMA VEZ, no final, no modelo escolhido


## 9. Modelo K=11 — design

Mesma estrutura hierarquica nao-centrada do Q8/Q9:
- `mu_alpha ~ N(0, 10)`
- `sigma_alpha ~ HalfNormal(10)`
- `mu_beta[k] ~ N(0, 2.5)` para k=0..10
- `sigma_beta[k] ~ HalfNormal(2.5)` para k=0..10
- `z_alpha[g] ~ N(0, 1)`, `z_beta[g,k] ~ N(0, 1)`
- `alpha_g = mu_alpha + sigma_alpha * z_alpha`
- `beta_g = mu_beta + sigma_beta * z_beta`
- `mu = alpha_g[g] + sum_k beta_g[g,k] * X[k]`
- Bernoulli: `p = sigmoid(mu)`, `y ~ Bernoulli(p)`

Parametros: 1 + 1 + 11 + 11 + 111 + 111*11 = **1.366** (vs 1.332 do K=10)


In [ ]:
# C10 — build_k11_model (parametrizada por n_features)
def build_k11_model(X, g_idx, n_generos, n_features, y_top25, feature_names, family='bernoulli'):
    coords = {'genero': genero_cats, 'feature': feature_names[:n_features]}
    with pm.Model(coords=coords) as model:
        X_data = pm.Data('X', X)
        g_data = pm.Data('genero_idx', g_idx)
        mu_alpha = pm.Normal('mu_alpha', mu=0.0, sigma=10.0)
        sigma_alpha = pm.HalfNormal('sigma_alpha', sigma=10.0)
        mu_beta = pm.Normal('mu_beta', mu=0.0, sigma=2.5, dims='feature')
        sigma_beta = pm.HalfNormal('sigma_beta', sigma=2.5, dims='feature')
        z_alpha = pm.Normal('z_alpha', mu=0.0, sigma=1.0, dims='genero')
        z_beta = pm.Normal('z_beta', mu=0.0, sigma=1.0, dims=('genero', 'feature'))
        alpha_g = pm.Deterministic('alpha_g', mu_alpha + sigma_alpha * z_alpha, dims='genero')
        beta_g = pm.Deterministic('beta_g', mu_beta + sigma_beta * z_beta, dims=('genero', 'feature'))
        mu = alpha_g[g_data] + (X_data * beta_g[g_data]).sum(axis=1)
        if family == 'bernoulli':
            pm.Bernoulli('y_obs', p=pm.math.sigmoid(mu), observed=y_top25)
        else:
            sigma_y = pm.HalfNormal('sigma_y', sigma=20.0)
            pm.Normal('y_obs', mu=mu, sigma=sigma_y, observed=y_top25)
    return model

print('build_k11_model pronto (parametrizado por n_features)')


In [ ]:
# C11 — Fit K=11 em TRAIN ONLY
print(f'[K=11] ADVI 20k em TRAIN ONLY n={len(X_train):,}, K={X_train.shape[1]}')
t0 = time.time()
with build_k11_model(X_train, g_train, n_generos, X_train.shape[1], y_train,
                    K11_FEATS, family='bernoulli'):
    approx_k11 = pm.fit(n=20_000, method='advi', random_seed=SEED,
                          progressbar=False,
                          obj_optimizer=pm.adam(learning_rate=5e-3))
idata_k11_train = approx_k11.sample(draws=2_000, random_seed=SEED)
elapsed_k11 = time.time() - t0
print(f'[K=11] concluido em {elapsed_k11/60:.1f} min')
idata_k11_train.to_netcdf(f'{RESULTS_DIR}/q10_k11_train.nc', engine='h5netcdf')
print('salvo: q10_k11_train.nc')


In [ ]:
# C12 — LOO do K=11 (in-sample, dentro do Train)
loo_k11 = az.loo(idata_k11_train, pointwise=True)
print('=== K=11 LOO (in-sample, dentro de Train) ===')
print(f'elpd_loo: {loo_k11.elpd_loo:.0f} (se={loo_k11.se:.0f})')
print(f'p_loo: {loo_k11.p_loo:.0f}')
n_pareto_bad = int((loo_k11.pareto_k > 0.7).sum())
print(f'# Pareto k > 0.7: {n_pareto_bad} de {len(loo_k11.pareto_k)} ({n_pareto_bad/len(loo_k11.pareto_k):.1%})')


## 13. K=10 baseline — fit em Train (mesma particao, para comparacao justa)

Usamos apenas as 10 colunas baseline (sem mode_bin). Mesmo Train set. Bernoulli ADVI.
**Por que refit K=10 em Train?** O K=10 existente (q9_baseline_bernoulli.nc) foi fitado em 90k, contaminado com Val+Test. Para comparar K=10 vs K=11 com rigor, ambos precisam ser fitados no mesmo Train.


In [ ]:
# C14 — Fit K=10 em TRAIN ONLY (X_train[:, :10], drop mode_bin)
X_train_k10 = X_train[:, :len(BASELINE_FEATS)]
print(f'[K=10] ADVI 20k em TRAIN ONLY n={len(X_train_k10):,}, K={X_train_k10.shape[1]}')
t0 = time.time()
with build_k11_model(X_train_k10, g_train, n_generos, X_train_k10.shape[1], y_train,
                    BASELINE_FEATS, family='bernoulli'):
    approx_k10 = pm.fit(n=20_000, method='advi', random_seed=SEED,
                          progressbar=False,
                          obj_optimizer=pm.adam(learning_rate=5e-3))
idata_k10_train = approx_k10.sample(draws=2_000, random_seed=SEED)
elapsed_k10 = time.time() - t0
print(f'[K=10] concluido em {elapsed_k10/60:.1f} min')
idata_k10_train.to_netcdf(f'{RESULTS_DIR}/q10_k10_train.nc', engine='h5netcdf')
print('salvo: q10_k10_train.nc')


In [ ]:
# C15 — LOO do K=10 (in-sample, dentro de Train)
loo_k10 = az.loo(idata_k10_train, pointwise=True)
print('=== K=10 LOO (in-sample, dentro de Train) ===')
print(f'elpd_loo: {loo_k10.elpd_loo:.0f} (se={loo_k10.se:.0f})')
print(f'p_loo: {loo_k10.p_loo:.0f}')
n_pareto_bad = int((loo_k10.pareto_k > 0.7).sum())
print(f'# Pareto k > 0.7: {n_pareto_bad} de {len(loo_k10.pareto_k)} ({n_pareto_bad/len(loo_k10.pareto_k):.1%})')


In [ ]:
# C16 — az.compare K=10 vs K=11 (LOO no mesmo Train)
comp = az.compare({'K10': idata_k10_train, 'K11': idata_k11_train})
print(comp)
comp.to_csv(f'{RESULTS_DIR}/q10_loo_comparison_train.csv')
print('salvo: q10_loo_comparison_train.csv')

elpd_diff = comp.loc['K11', 'elpd_diff'] - comp.loc['K10', 'elpd_diff']
dse = comp.loc['K11', 'dse']
print(f'\nK=11 - K=10 (LOO): elpd_diff = {elpd_diff:.0f}, dse = {dse:.0f}, t = {elpd_diff/dse:.1f}')
if elpd_diff > 2 * dse:
    print('LOO: K=11 GANHA no train')
elif elpd_diff < -2 * dse:
    print('LOO: K=11 PERDE no train')
else:
    print('LOO: K=11 EMPATA com K=10 no train')


In [ ]:
# C17 — Validacao em VAL: AUC, Brier, log-loss para K=10 e K=11
def predict_p_hit(idata, X, g_idx):
    """Prediz P(hit) usando medias pontuais de alpha_g e beta_g."""
    mu_alpha = float(idata.posterior['mu_alpha'].mean())
    sigma_alpha = float(idata.posterior['sigma_alpha'].mean())
    mu_beta = idata.posterior['mu_beta'].mean(dim=('chain', 'draw')).values
    sigma_beta = idata.posterior['sigma_beta'].mean(dim=('chain', 'draw')).values
    alpha_g = idata.posterior['alpha_g'].mean(dim=('chain', 'draw')).values
    beta_g = idata.posterior['beta_g'].mean(dim=('chain', 'draw')).values
    mu = alpha_g[g_idx] + (X * beta_g[g_idx]).sum(axis=1)
    return 1.0 / (1.0 + np.exp(-mu))

print('=== Metricas em VAL (K=10 e K=11) ===')
p_val_k10 = predict_p_hit(idata_k10_train, X_val[:, :len(BASELINE_FEATS)], g_val)
p_val_k11 = predict_p_hit(idata_k11_train, X_val, g_val)

metrics = {}
for name, p in [('K=10', p_val_k10), ('K=11', p_val_k11)]:
    auc = roc_auc_score(y_val, p)
    brier = brier_score_loss(y_val, p)
    ll = log_loss(y_val, np.clip(p, 1e-7, 1-1e-7))
    acc = float(((p > 0.5).astype(int) == y_val).mean())
    metrics[name] = {'auc': auc, 'brier': brier, 'log_loss': ll, 'accuracy': acc}
    print(f'{name}: AUC={auc:.3f}, Brier={brier:.3f}, log-loss={ll:.3f}, acc={acc:.3f}')

# Decision: K=11 wins if BOTH LOO and AUC better
loo_k11_wins = elpd_diff > 2 * dse
auc_k11_wins = metrics['K=11']['auc'] > metrics['K=10']['auc']
brier_k11_wins = metrics['K=11']['brier'] < metrics['K=10']['brier']

print(f'\nLOO: K=11 wins? {loo_k11_wins}')
print(f'AUC (Val): K=11 wins? {auc_k11_wins} ({metrics["K=11"]["auc"]:.3f} vs {metrics["K=10"]["auc"]:.3f})')
print(f'Brier (Val): K=11 wins? {brier_k11_wins} ({metrics["K=11"]["brier"]:.3f} vs {metrics["K=10"]["brier"]:.3f})')

if loo_k11_wins and (auc_k11_wins or brier_k11_wins):
    CHOSEN = 'K=11'
    print('\nDECISAO: K=11 escolhido (LOO e ao menos 1 metrica de Val melhor)')
else:
    CHOSEN = 'K=10'
    print('\nDECISAO: K=10 escolhido (K=11 nao generaliza)')

CHOSEN_IDATA = idata_k11_train if CHOSEN == 'K=11' else idata_k10_train
CHOSEN_FEATS = K11_FEATS if CHOSEN == 'K=11' else BASELINE_FEATS
print(f'Chosen: {CHOSEN} com {len(CHOSEN_FEATS)} features')


## 18. Test set — avaliacao FINAL do modelo escolhido

**REGRA:** Test set NAO foi tocado ate agora. Agora avaliamos o modelo escolhido UMA vez. NAO ajustar nada com base no resultado de Test.

Metricas reportadas: AUC, Brier, log-loss, accuracy.
Tambem: per-genre (top 10 generos) e calibracao (reliability table).


In [ ]:
# C19 — Predicao em TEST (X_test) para o modelo escolhido
if CHOSEN == 'K=11':
    X_test_chosen = X_test
else:
    X_test_chosen = X_test[:, :len(BASELINE_FEATS)]

p_test = predict_p_hit(CHOSEN_IDATA, X_test_chosen, g_test)
auc_test = roc_auc_score(y_test, p_test)
brier_test = brier_score_loss(y_test, p_test)
ll_test = log_loss(y_test, np.clip(p_test, 1e-7, 1-1e-7))
acc_test = float(((p_test > 0.5).astype(int) == y_test).mean())

print(f'=== TEST SET — modelo escolhido: {CHOSEN} ===')
print(f'n_test: {len(y_test):,}')
print(f'AUC:       {auc_test:.3f}')
print(f'Brier:     {brier_test:.3f}')
print(f'Log-loss:  {ll_test:.3f}')
print(f'Accuracy:  {acc_test:.3f}')

# Baseline (prevalencia) para referencia
prevalence = float(y_test.mean())
brier_baseline = prevalence * (1 - prevalence)
print(f'\nBaseline (prevalencia = {prevalence:.3f}):')
print(f'  Brier baseline = {brier_baseline:.3f}')
print(f'  Brier skill score = {1 - brier_test/brier_baseline:.3f} (positivo = melhor que baseline)')


In [ ]:
# C20 — Per-genre metrics em TEST (top 10 generos)
from collections import Counter
gen_counts = Counter(g_test)
top_10_generos = [g for g, _ in gen_counts.most_common(10)]
top_10_names = [genero_cats[g] for g in top_10_generos]
print(f'Top 10 generos em Test: {top_10_names}')

per_genre_rows = []
for g, name in zip(top_10_generos, top_10_names):
    mask = g_test == g
    n_g = int(mask.sum())
    if n_g < 50:
        continue
    auc_g = roc_auc_score(y_test[mask], p_test[mask]) if len(set(y_test[mask])) > 1 else float('nan')
    brier_g = brier_score_loss(y_test[mask], p_test[mask])
    per_genre_rows.append({'genero': name, 'n': n_g, 'auc': auc_g, 'brier': brier_g})
    print(f'  {name:20s}  n={n_g:5d}  AUC={auc_g:.3f}  Brier={brier_g:.3f}')

per_genre_df = pd.DataFrame(per_genre_rows)
per_genre_df.to_csv(f'{RESULTS_DIR}/q10_per_genre_test.csv', index=False)
print('salvo: q10_per_genre_test.csv')


In [ ]:
# C21 — Calibracao em TEST (reliability diagram + ECE)
n_bins = 10
bins = np.linspace(0, 1, n_bins + 1)
bin_idx = np.digitize(p_test, bins) - 1
bin_idx = np.clip(bin_idx, 0, n_bins - 1)

calib_rows = []
ece = 0.0
for b in range(n_bins):
    mask = bin_idx == b
    n_b = int(mask.sum())
    if n_b == 0:
        continue
    p_mean = float(p_test[mask].mean())
    y_mean = float(y_test[mask].mean())
    ece += abs(p_mean - y_mean) * n_b / len(y_test)
    calib_rows.append({'bin': b, 'lo': bins[b], 'hi': bins[b+1],
                       'n': n_b, 'p_pred_mean': p_mean, 'y_actual': y_mean})

calib_df = pd.DataFrame(calib_rows)
print('=== Calibracao em Test ===')
print(calib_df.to_string(index=False))
print(f'\nECE (Expected Calibration Error): {ece:.3f} (alvo: < 0.05)')
calib_df.to_csv(f'{RESULTS_DIR}/q10_calibration_test.csv', index=False)
print('salvo: q10_calibration_test.csv')


In [ ]:
# C22 — Tempo de inferencia por faixa em TEST
t0 = time.time()
_ = predict_p_hit(CHOSEN_IDATA, X_test_chosen[:1000], g_test[:1000])
t1 = time.time()
ms_per_track = (t1 - t0) / 1000 * 1000
print(f'Tempo de inferencia: {ms_per_track:.1f} ms por faixa (1000 chamadas)')
print(f'Alvo para produto: < 100 ms (UX sem espera)')
if ms_per_track < 100:
    print('OK para tempo real.')
else:
    print('LENTO — pode precisar de simplificacao no produto.')


In [ ]:
# C23 — Salva todos os artefatos para o produto
import json

summary = pd.DataFrame([
    {'model': 'K=10', 'K': len(BASELINE_FEATS), 'LOO_elpd': loo_k10.elpd_loo, 'LOO_se': loo_k10.se,
     'Val_AUC': metrics['K=10']['auc'], 'Val_Brier': metrics['K=10']['brier']},
    {'model': 'K=11', 'K': len(K11_FEATS), 'LOO_elpd': loo_k11.elpd_loo, 'LOO_se': loo_k11.se,
     'Val_AUC': metrics['K=11']['auc'], 'Val_Brier': metrics['K=11']['brier']},
    {'model': 'CHOSEN', 'K': len(CHOSEN_FEATS), 'Test_AUC': auc_test, 'Test_Brier': brier_test,
     'Test_LogLoss': ll_test, 'Test_Accuracy': acc_test, 'ECE': ece, 'ms_per_track': ms_per_track},
])
summary.to_csv(f'{RESULTS_DIR}/q10_summary.csv', index=False)
print('salvo: q10_summary.csv')
print(summary.to_string(index=False))

# Salva parametros do modelo escolhido (para deploy)
mu_alpha_s = float(CHOSEN_IDATA.posterior['mu_alpha'].mean())
sigma_alpha_s = float(CHOSEN_IDATA.posterior['sigma_alpha'].mean())
mu_beta_s = CHOSEN_IDATA.posterior['mu_beta'].mean(dim=('chain', 'draw')).values.tolist()
sigma_beta_s = CHOSEN_IDATA.posterior['sigma_beta'].mean(dim=('chain', 'draw')).values.tolist()
alpha_g_s = CHOSEN_IDATA.posterior['alpha_g'].mean(dim=('chain', 'draw')).values.tolist()
beta_g_s = CHOSEN_IDATA.posterior['beta_g'].mean(dim=('chain', 'draw')).values.tolist()

prod_artifacts = {
    'chosen_model': CHOSEN,
    'feature_names': CHOSEN_FEATS,
    'mu_alpha': mu_alpha_s,
    'sigma_alpha': sigma_alpha_s,
    'mu_beta': mu_beta_s,
    'sigma_beta': sigma_beta_s,
    'alpha_g': alpha_g_s,
    'beta_g': beta_g_s,
    'genero_cats': genero_cats,
    'feat_means': means.to_dict(),
    'feat_stds': stds.to_dict(),
    'test_metrics': {'auc': float(auc_test), 'brier': float(brier_test),
                     'log_loss': float(ll_test), 'accuracy': float(acc_test)},
    'val_metrics': metrics,
    'inference_ms': float(ms_per_track),
    'ece': float(ece),
    'n_train': int(len(X_train)), 'n_val': int(len(X_val)), 'n_test': int(len(X_test)),
}
with open(f'{RESULTS_DIR}/q10_prod_artifacts.json', 'w', encoding='utf-8') as f:
    json.dump(prod_artifacts, f, indent=2, ensure_ascii=False)
print('salvo: q10_prod_artifacts.json (parametros para deploy)')


## 24. Recomendacao final

**Modelo escolhido:** {CHOSEN} (declarado em C17)

**Metricas em Test (avaliacao final):**
- AUC: ver C19 (alvo > 0.65)
- Brier: ver C19 (alvo < 0.18)
- ECE: ver C21 (alvo < 0.05)
- Tempo de inferencia: ver C22 (alvo < 100ms)

**Para o produto:**
- Backend recebe JSON com `track_features` e `genero` (do dropdown de 111)
- Retorna `score` (0-100) + 2-3 frases de explicacao baseadas em beta_g
- Artefatos em `q10_prod_artifacts.json` ja contem todos os parametros

**Proximos passos:**
1. Se K=11 venceu: usar `q10_prod_artifacts.json` (que ja tem mode_bin)
2. Se K=10 venceu: o produto usa o modelo Q8 original (ja tem artefatos em q8_coefs_globais.csv)
3. Implementar endpoint backend (FastAPI ou Flask) com lookup por genero_idx
4. Implementar frontend com dropdown de 111 generos
5. Validar UX com 5-10 musicas reais

**Caveats:**
- ADVI tem geometria melhor que NUTS mas pode ter multiplos modais
- Modelo assume genero conhecido (sem desambiguacao automatica)
- Scores sao calibrados em popularidade do Spotify (0-100), nao sao 'qualidade musical'
